# End-to-End Notebook Run (Stages 1-4)

This notebook replaces manual CLI orchestration for:
1. encoder preparation checks (Stage 1)
2. fixed split preparation (Stages 2-3)
3. downstream training runs (Stage 4)
4. run-table and seed-summary reporting

It uses the same core logic in `src/cv/` used by scripts.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.hub import download_url_to_file
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if not SRC_ROOT.exists():
    raise FileNotFoundError(f"Could not find src/ under {PROJECT_ROOT}")

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from cv.config import ARTIFACTS_ROOT, DEFAULT_ENCODER_CHECKPOINTS
from cv.data import build_downstream_datasets, create_fixed_split_indices, load_fixed_split_indices
from cv.encoders import load_encoder
from cv.explain import (
    discover_stage4_runs,
    generate_explanations_for_runs,
    run_explanation_qc,
    write_explanation_qc_report,
)
from cv.models import build_downstream_model, resolve_mode_config
from cv.train.metrics import summarize_test_accuracy_by_condition
from cv.train.trainer import (
    TrainingRunConfig,
    build_run_table_row,
    resolve_training_recipe,
    train_one_run,
)
from cv.utils.io import ensure_parent, read_json, write_json

## Runtime Configuration

Set switches here. `run_training=False` by default to avoid accidental long runs.

In [ ]:
RUN_CONFIG: dict[str, object] = {
    # Scope
    "conditions": ["supervised", "moco", "swav", "random_init"],
    "seeds": [0, 1, 2],

    # Device + loaders
    "device": "cpu",
    "num_workers": 0,
    "pin_memory": False,

    # Data / artifacts
    "artifacts_root": None,
    "data_root": None,
    "download_data": False,

    # Stage 1
    "prepare_checkpoints": False,
    "force_download_checkpoints": False,
    "inspect_encoders": True,
    "allow_remote_download": False,

    # Stages 2-3
    "create_splits": True,
    "overwrite_splits": False,
    "split_seed": 42,
    "val_ratio": 0.2,

    # Stage 4
    "run_cross_condition_sanity_check": True,
    "run_training": False,
    "sanity_checks": True,

    # Optional recipe overrides (None = use fixed defaults by condition)
    "probe_recipe_id": None,
    "random_init_recipe_id": None,
}

ARTIFACTS_ROOT_RESOLVED = Path(RUN_CONFIG["artifacts_root"]) if RUN_CONFIG["artifacts_root"] else ARTIFACTS_ROOT
print(json.dumps(RUN_CONFIG, indent=2))
print(f"Resolved artifacts root: {ARTIFACTS_ROOT_RESOLVED}")

## Stage 1 - Encoder Preparation and Inspection

In [ ]:
ENCODER_CONDITIONS = ["supervised", "moco", "swav"]

def _download_checkpoint(url: str, destination: Path, *, force: bool) -> None:
    if destination.exists() and not force:
        print(f"[skip] exists: {destination}")
        return

    ensure_parent(destination)
    print(f"[download] {url} -> {destination}")
    download_url_to_file(url, str(destination), progress=True)


def prepare_encoder_checkpoints(*, force_download: bool) -> None:
    cfg = DEFAULT_ENCODER_CHECKPOINTS
    _download_checkpoint(
        cfg.moco_checkpoint_url,
        cfg.moco_checkpoint_path,
        force=force_download,
    )
    _download_checkpoint(
        cfg.swav_checkpoint_url,
        cfg.swav_checkpoint_path,
        force=force_download,
    )


def inspect_encoders(
    *,
    conditions: list[str],
    device: str,
    allow_remote_download: bool,
    batch_size: int = 2,
) -> tuple[pd.DataFrame, dict[str, object]]:
    rows: list[dict[str, object]] = []
    failures: list[dict[str, str]] = []
    torch_device = torch.device(device)

    for condition in conditions:
        kwargs: dict[str, object] = {"freeze": True, "device": torch_device}
        if condition == "moco":
            kwargs["checkpoint_path"] = str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path)
            kwargs["allow_remote_download"] = allow_remote_download
        elif condition == "swav":
            kwargs["checkpoint_path"] = str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path)
            kwargs["allow_remote_download"] = allow_remote_download

        try:
            loaded = load_encoder(condition, **kwargs)
            images = torch.randn(batch_size, 3, 224, 224, device=torch_device)
            with torch.no_grad():
                features = loaded.encoder(images)

            if tuple(features.shape) != (batch_size, loaded.metadata.feature_dim):
                raise ValueError(
                    f"Unexpected feature shape for {condition}: {tuple(features.shape)}"
                )

            row = {
                "condition": condition,
                "feature_shape": list(features.shape),
                "feature_dim": loaded.metadata.feature_dim,
                "gradcam_layer": loaded.metadata.gradcam_target_layer,
                "encoder_frozen": not any(p.requires_grad for p in loaded.encoder.parameters()),
                "checkpoint_id": loaded.metadata.checkpoint_id,
                "checkpoint_origin": loaded.metadata.checkpoint_origin,
                "pretraining_objective": loaded.metadata.pretraining_objective,
            }
            rows.append(row)
        except Exception as exc:
            failures.append({"condition": condition, "error": str(exc)})

    report = {
        "stage": "stage-1-encoder-preparation",
        "device": device,
        "results": rows,
        "failures": failures,
    }
    return pd.DataFrame(rows), report


if RUN_CONFIG["prepare_checkpoints"]:
    prepare_encoder_checkpoints(force_download=bool(RUN_CONFIG["force_download_checkpoints"]))
else:
    print("Skipping checkpoint downloads (prepare_checkpoints=False).")

if RUN_CONFIG["inspect_encoders"]:
    encoder_df, encoder_report = inspect_encoders(
        conditions=ENCODER_CONDITIONS,
        device=str(RUN_CONFIG["device"]),
        allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
    )
    display(encoder_df)

    encoder_report_path = ARTIFACTS_ROOT_RESOLVED / "metrics" / "encoder_prep_report_notebook.json"
    write_json(encoder_report_path, encoder_report)
    print(f"Saved encoder report to: {encoder_report_path}")

    if encoder_report["failures"]:
        raise RuntimeError(f"Encoder inspection failures: {encoder_report['failures']}")
else:
    print("Skipping encoder inspection (inspect_encoders=False).")

## Stages 2-3 - Fixed Split and Data Sanity Checks

In [ ]:
if RUN_CONFIG["create_splits"]:
    split_artifacts = create_fixed_split_indices(
        data_root=RUN_CONFIG["data_root"],
        artifacts_root=RUN_CONFIG["artifacts_root"],
        split_seed=int(RUN_CONFIG["split_seed"]),
        val_ratio=float(RUN_CONFIG["val_ratio"]),
        overwrite=bool(RUN_CONFIG["overwrite_splits"]),
        download=bool(RUN_CONFIG["download_data"]),
    )
else:
    split_artifacts = load_fixed_split_indices(artifacts_root=RUN_CONFIG["artifacts_root"])

print("Split artifacts:")
print(f"- train indices: {split_artifacts.train_indices_path}")
print(f"- val indices:   {split_artifacts.val_indices_path}")
print(f"- metadata:      {split_artifacts.metadata_path}")

split_metadata = split_artifacts.metadata
display(pd.DataFrame([split_metadata]))

class_counts = split_metadata.get("class_counts", {})
if isinstance(class_counts, dict) and class_counts:
    class_counts_df = pd.DataFrame(class_counts).fillna(0).astype(int)
    display(class_counts_df)

datasets = build_downstream_datasets(
    train_indices=split_artifacts.train_indices,
    val_indices=split_artifacts.val_indices,
    data_root=RUN_CONFIG["data_root"],
    download=bool(RUN_CONFIG["download_data"]),
)

batch = next(iter(DataLoader(datasets.train, batch_size=8, shuffle=False, num_workers=0)))
images, labels = batch
print(f"Train sample batch image shape: {tuple(images.shape)}")
print(f"Train sample batch label range: [{int(labels.min())}, {int(labels.max())}]")
print(f"Train subset size: {len(datasets.train)}")
print(f"Val subset size:   {len(datasets.val)}")
print(f"Test set size:     {len(datasets.test)}")

## Stage 4 - Optional Cross-Condition Check and Training Runs

In [ ]:
def _recipe_id_for_condition(condition: str) -> str | None:
    if condition == "random_init":
        return RUN_CONFIG["random_init_recipe_id"]
    return RUN_CONFIG["probe_recipe_id"]


def run_cross_condition_one_batch_check(*, conditions: list[str]) -> None:
    device = torch.device(str(RUN_CONFIG["device"]))
    batch_loader = DataLoader(datasets.train, batch_size=8, shuffle=False, num_workers=0)
    images, targets = next(iter(batch_loader))
    images = images.to(device)

    labels_min = int(targets.min().item())
    labels_max = int(targets.max().item())
    if labels_min < 0 or labels_max > 9:
        raise RuntimeError(
            f"Unexpected label range: min={labels_min}, max={labels_max}"
        )

    for condition in conditions:
        recipe = resolve_training_recipe(
            condition=condition,
            recipe_id=_recipe_id_for_condition(condition),
        )
        mode_config = resolve_mode_config(
            condition=condition,
            training_mode=recipe.training_mode,
        )

        checkpoint_path = None
        if condition == "moco":
            checkpoint_path = str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path)
        elif condition == "swav":
            checkpoint_path = str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path)

        model = build_downstream_model(
            condition=condition,
            num_classes=10,
            freeze_encoder=mode_config.freeze_encoder,
            trainable_layer4=mode_config.trainable_layer4,
            device=device,
            allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
            checkpoint_path=checkpoint_path,
        )
        model.eval()
        with torch.no_grad():
            logits = model(images)

        expected_shape = (images.shape[0], 10)
        if tuple(logits.shape) != expected_shape:
            raise RuntimeError(
                f"Unexpected logits shape for {condition}: {tuple(logits.shape)}"
            )


if RUN_CONFIG["run_cross_condition_sanity_check"]:
    run_cross_condition_one_batch_check(conditions=list(RUN_CONFIG["conditions"]))
    print("Cross-condition one-batch sanity check passed.")
else:
    print("Skipping cross-condition one-batch sanity check.")

run_results: list[dict[str, object]] = []
if RUN_CONFIG["run_training"]:
    for condition in list(RUN_CONFIG["conditions"]):
        recipe_id = _recipe_id_for_condition(condition)
        for seed in list(RUN_CONFIG["seeds"]):
            config = TrainingRunConfig(
                condition=condition,
                seed=int(seed),
                recipe_id=recipe_id,
                device=str(RUN_CONFIG["device"]),
                artifacts_root=RUN_CONFIG["artifacts_root"],
                data_root=RUN_CONFIG["data_root"],
                download=bool(RUN_CONFIG["download_data"]),
                num_workers=int(RUN_CONFIG["num_workers"]),
                pin_memory=bool(RUN_CONFIG["pin_memory"]),
                allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
                moco_checkpoint_path=str(DEFAULT_ENCODER_CHECKPOINTS.moco_checkpoint_path),
                swav_checkpoint_path=str(DEFAULT_ENCODER_CHECKPOINTS.swav_checkpoint_path),
                sanity_checks=bool(RUN_CONFIG["sanity_checks"]),
            )
            print(f"[run] condition={condition} seed={seed}")
            result = train_one_run(config)
            run_results.append(result)
            print(
                f"[done] condition={condition} seed={seed} "
                f"best_val_acc={result['best_val_acc']:.4f} test_acc={result['test_acc']:.4f}"
            )
else:
    print("Skipping training (run_training=False).")

## Load Existing Run Metrics, Summarize, and Plot

In [ ]:
def load_existing_run_results(
    *,
    run_metrics_root: Path,
    conditions: list[str],
    seeds: list[int],
) -> list[dict[str, object]]:
    if not run_metrics_root.exists():
        return []

    allowed_conditions = set(conditions)
    allowed_seeds = set(seeds)
    payloads: list[dict[str, object]] = []

    for condition_dir in sorted(run_metrics_root.glob("*")):
        if not condition_dir.is_dir():
            continue
        condition = condition_dir.name
        if condition not in allowed_conditions:
            continue

        for json_path in sorted(condition_dir.glob("*.json")):
            payload = read_json(json_path)
            if not isinstance(payload, dict):
                continue
            seed = payload.get("seed")
            if not isinstance(seed, int):
                continue
            if seed not in allowed_seeds:
                continue
            payloads.append(payload)

    return payloads


run_metrics_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "probe_runs"
existing_results = load_existing_run_results(
    run_metrics_root=run_metrics_root,
    conditions=list(RUN_CONFIG["conditions"]),
    seeds=[int(seed) for seed in list(RUN_CONFIG["seeds"])],
)

# Prefer fresh in-memory results when a condition+seed key overlaps.
combined_by_key: dict[tuple[str, int], dict[str, object]] = {}
for payload in existing_results:
    key = (str(payload["condition"]), int(payload["seed"]))
    combined_by_key[key] = payload
for payload in run_results:
    key = (str(payload["condition"]), int(payload["seed"]))
    combined_by_key[key] = payload

combined_results = [combined_by_key[key] for key in sorted(combined_by_key)]
if not combined_results:
    print("No run results found. Run training or point to existing artifacts first.")
else:
    run_rows = [build_run_table_row(payload) for payload in combined_results]
    run_df = pd.DataFrame(run_rows).sort_values(["condition", "seed"]).reset_index(drop=True)
    display(run_df)

    summaries = summarize_test_accuracy_by_condition(run_rows)
    summary_df = pd.DataFrame(summaries).sort_values(["condition"]).reset_index(drop=True)
    display(summary_df)

    output_root = ARTIFACTS_ROOT_RESOLVED / "metrics" / "probe_runs"
    ensure_parent(output_root / "placeholder.txt")

    run_table_json = output_root / "run_table.notebook.json"
    run_table_csv = output_root / "run_table.notebook.csv"
    summary_json = output_root / "condition_summary.notebook.json"
    summary_csv = output_root / "condition_summary.notebook.csv"

    write_json(run_table_json, run_rows)
    run_df.to_csv(run_table_csv, index=False)
    write_json(summary_json, summaries)
    summary_df.to_csv(summary_csv, index=False)

    print(f"Saved run table JSON: {run_table_json}")
    print(f"Saved run table CSV:  {run_table_csv}")
    print(f"Saved summary JSON:   {summary_json}")
    print(f"Saved summary CSV:    {summary_csv}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(summary_df["condition"], summary_df["mean_test_acc"], yerr=summary_df["std_test_acc"], capsize=4)
    ax.set_title("Mean +/- Std Test Accuracy by Condition")
    ax.set_xlabel("Condition")
    ax.set_ylabel("Test Top-1 Accuracy")
    ax.set_ylim(0.0, 1.0)
    ax.grid(axis="y", alpha=0.25)
    plt.show()

## Stages 5-6 - Generate Explanations and Run QC

In [ ]:
EXPLAIN_CONFIG: dict[str, object] = {
    "run_explanations": False,
    "run_qc": False,
    "conditions": list(RUN_CONFIG["conditions"]),
    "seeds": [int(seed) for seed in list(RUN_CONFIG["seeds"])],
    "methods": ["gradcam", "gradcampp", "occlusion"],
    "batch_size": 8,
    "overwrite": False,
}

print(json.dumps(EXPLAIN_CONFIG, indent=2))

if EXPLAIN_CONFIG["run_explanations"]:
    runs = discover_stage4_runs(
        artifacts_root=RUN_CONFIG["artifacts_root"],
        conditions=list(EXPLAIN_CONFIG["conditions"]),
        seeds=list(EXPLAIN_CONFIG["seeds"]),
    )
    saliency_rows = generate_explanations_for_runs(
        runs=runs,
        methods=list(EXPLAIN_CONFIG["methods"]),
        batch_size=int(EXPLAIN_CONFIG["batch_size"]),
        data_root=RUN_CONFIG["data_root"],
        artifacts_root=RUN_CONFIG["artifacts_root"],
        device=str(RUN_CONFIG["device"]),
        allow_remote_download=bool(RUN_CONFIG["allow_remote_download"]),
        overwrite=bool(EXPLAIN_CONFIG["overwrite"]),
        download=bool(RUN_CONFIG["download_data"]),
    )
    print(f"Generated saliency rows: {len(saliency_rows)}")
else:
    print("Skipping explanation generation (run_explanations=False).")

if EXPLAIN_CONFIG["run_qc"]:
    qc_report = run_explanation_qc(
        artifacts_root=RUN_CONFIG["artifacts_root"],
        conditions=list(EXPLAIN_CONFIG["conditions"]),
        seeds=list(EXPLAIN_CONFIG["seeds"]),
        methods=list(EXPLAIN_CONFIG["methods"]),
    )
    qc_report_path = write_explanation_qc_report(
        report=qc_report,
        artifacts_root=RUN_CONFIG["artifacts_root"],
    )
    print(f"Saved QC report: {qc_report_path}")
    display(pd.DataFrame(qc_report.get("coverage", [])))
    if qc_report.get("errors"):
        display(pd.DataFrame(qc_report["errors"]))
else:
    print("Skipping explanation QC (run_qc=False).")